# Fake Reviews Dataset Baseline Models

Compares two feature approaches for fake-review classification:

1. **Engineered features** - uses the linguistic feature table from the Feature Engineering notebook (`word_count`, `readability`, `sentiment`, `POS ratios`, etc.). Models: Logistic Regression, Random Forest.
2. **TF-IDF on raw text** - classic bag-of-words baseline straight from `review_text`. Models: Logistic Regression, Linear SVM.

Run **both modes** and compare - this comparison is exactly the kind of "justification of technology choices" evidence your report needs, since you can show numerically whether hand-engineered linguistic signal or raw text patterns carry more predictive power.

| Mode | Input |
|---|---|
| `features` | Output of the Feature Engineering notebook |
| `tfidf` | Output of the Clean & EDA notebook (needs `review_text`) |

## Setup

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

RANDOM_STATE = 42


## Shared Evaluation Helper

In [2]:
def evaluate(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label="fake")
    rec = recall_score(y_true, y_pred, pos_label="fake")
    f1 = f1_score(y_true, y_pred, pos_label="fake")
    cm = confusion_matrix(y_true, y_pred, labels=["real", "fake"])

    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(f"Accuracy:            {acc:.4f}")
    print(f"Precision (fake):    {prec:.4f}")
    print(f"Recall (fake):       {rec:.4f}   <- most important: cost of MISSING a fake review")
    print(f"F1 (fake):           {f1:.4f}")
    print("Confusion matrix [rows=actual, cols=predicted] order=[real, fake]:")
    print(cm)
    print()
    print(classification_report(y_true, y_pred, digits=4))

    return {"model": name, "accuracy": acc, "precision_fake": prec, "recall_fake": rec, "f1_fake": f1}


## Mode 1: Engineered Features

Logistic Regression and Random Forest trained on the hand-engineered linguistic feature table.

In [ ]:
def run_features_mode(path):
    df = pd.read_csv(path)
    print(f"Loaded {len(df)} rows, {df.shape[1]} columns from {path}")

    # Use only numeric engineered features (drop rating/category - categorical/context, not linguistic)
    feature_cols = [
        "word_count", "char_count", "avg_word_length", "sentence_count",
        "avg_sentence_length", "flesch_reading_ease", "flesch_kincaid_grade",
        "unique_word_ratio", "exclamation_ratio", "capital_word_ratio",
        "punctuation_density", "sentiment_polarity", "sentiment_subjectivity",
        "adjective_ratio", "noun_ratio", "verb_ratio",
    ]
    feature_cols = [c for c in feature_cols if c in df.columns]
    print(f"Using {len(feature_cols)} engineered features: {feature_cols}")

    df = df.dropna(subset=feature_cols + ["label_clean"])
    X = df[feature_cols].values
    y = df["label_clean"].values

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    results = []

    # Logistic Regression
    lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    lr.fit(X_train_s, y_train)
    results.append(evaluate("Engineered Features - Logistic Regression", y_test, lr.predict(X_test_s)))

    # Random Forest (no scaling needed, but harmless to reuse scaled data)
    rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(X_train, y_train)
    results.append(evaluate("Engineered Features - Random Forest", y_test, rf.predict(X_test)))

    # Feature importance from Random Forest - useful for your Results chapter
    importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
    print("Random Forest feature importances (most predictive first):")
    print(importances.round(4))

    return results


Run engineered-features mode (edit the path to point at your feature-engineering output):

In [4]:
FEATURES_INPUT_PATH = "data/processed/fake_reviews_features.csv"
features_results = run_features_mode(FEATURES_INPUT_PATH)


Loaded 40405 rows, 19 columns from data/processed/fake_reviews_features.csv
Using 16 engineered features: ['word_count', 'char_count', 'avg_word_length', 'sentence_count', 'avg_sentence_length', 'flesch_reading_ease', 'flesch_kincaid_grade', 'unique_word_ratio', 'exclamation_ratio', 'capital_word_ratio', 'punctuation_density', 'sentiment_polarity', 'sentiment_subjectivity', 'adjective_ratio', 'noun_ratio', 'verb_ratio']

  Engineered Features - Logistic Regression
Accuracy:            0.7836
Precision (fake):    0.7873
Recall (fake):       0.7766   <- most important: cost of MISSING a fake review
F1 (fake):           0.7819
Confusion matrix [rows=actual, cols=predicted] order=[real, fake]:
[[3196  847]
 [ 902 3136]]

              precision    recall  f1-score   support

        fake     0.7873    0.7766    0.7819      4038
        real     0.7799    0.7905    0.7852      4043

    accuracy                         0.7836      8081
   macro avg     0.7836    0.7836    0.7836      8081
w

## Mode 2: TF-IDF on Raw Text

Logistic Regression and Linear SVM trained on a TF-IDF bag-of-words representation of `review_text`.

In [ ]:
def run_tfidf_mode(path):
    df = pd.read_csv(path)
    print(f"Loaded {len(df)} rows from {path}")

    df = df.dropna(subset=["review_text", "label_clean"])
    X_text = df["review_text"].astype(str).values
    y = df["label_clean"].values

    X_train_text, X_test_text, y_train, y_test = train_test_split(X_text, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

    vectorizer = TfidfVectorizer(
        max_features=5000, ngram_range=(1, 2), min_df=2, stop_words="english"
    )
    X_train = vectorizer.fit_transform(X_train_text)
    X_test = vectorizer.transform(X_test_text)

    results = []

    lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    lr.fit(X_train, y_train)
    results.append(evaluate("TF-IDF - Logistic Regression", y_test, lr.predict(X_test)))

    svm = LinearSVC(random_state=RANDOM_STATE)
    svm.fit(X_train, y_train)
    results.append(evaluate("TF-IDF - Linear SVM", y_test, svm.predict(X_test)))

    # Most predictive n-grams - great qualitative evidence for your report
    feature_names = np.array(vectorizer.get_feature_names_out())
    coefs = lr.coef_[0]
    top_fake = feature_names[np.argsort(coefs)[-15:]][::-1]
    top_real = feature_names[np.argsort(coefs)[:15]]
    print("Top terms associated with FAKE reviews:", list(top_fake))
    print("Top terms associated with REAL reviews:", list(top_real))

    return results


Run TF-IDF mode (edit the path to point at your cleaned dataset):

In [ ]:
TFIDF_INPUT_PATH = "data/processed/fake_reviews_cleaned.csv"
tfidf_results = run_tfidf_mode(TFIDF_INPUT_PATH)

Loaded 40405 rows from data/processed/fake_reviews_cleaned.csv

  TF-IDF - Logistic Regression
Accuracy:            0.8776
Precision (fake):    0.8789
Recall (fake):       0.8757   <- most important: cost of MISSING a fake review
F1 (fake):           0.8773
Confusion matrix [rows=actual, cols=predicted] order=[real, fake]:
[[3556  487]
 [ 502 3536]]

              precision    recall  f1-score   support

        fake     0.8789    0.8757    0.8773      4038
        real     0.8763    0.8795    0.8779      4043

    accuracy                         0.8776      8081
   macro avg     0.8776    0.8776    0.8776      8081
weighted avg     0.8776    0.8776    0.8776      8081


  TF-IDF - Linear SVM
Accuracy:            0.8839
Precision (fake):    0.8825
Recall (fake):       0.8856   <- most important: cost of MISSING a fake review
F1 (fake):           0.8841
Confusion matrix [rows=actual, cols=predicted] order=[real, fake]:
[[3567  476]
 [ 462 3576]]

              precision    recall  f1-s

## Summary

Combine results from both modes into a single comparison table.

In [9]:
all_results = features_results + tfidf_results
summary = pd.DataFrame(all_results)
summary


,model,accuracy,precision_fake,recall_fake,f1_fake
0,Engineered Features - Logistic Regression,0.783566,0.787346,0.776622,0.781947
1,Engineered Features - Random Forest,0.851627,0.853902,0.848192,0.851037
2,TF-IDF - Logistic Regression,0.877614,0.878946,0.875681,0.877311
3,TF-IDF - Linear SVM,0.883925,0.882527,0.885587,0.884054


In [8]:
cleaned = pd.read_csv("data/processed/fake_reviews_cleaned.csv")
feats = pd.read_csv("data/processed/fake_reviews_features_full.csv")
merged = pd.concat([cleaned[["review_text"]].reset_index(drop=True), feats.reset_index(drop=True)], axis=1)
merged.to_csv("data/processed/fake_reviews_merged.csv", index=False)